In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT / "src"))

import pandas as pd

from healthllm.llm_client import get_ollama_llm
from healthllm.evaluate import run_readiness_evaluation, compute_readiness_metrics


In [4]:
PROJECT_ROOT = Path.cwd().parent
PROCESSED_DATA_DIR = PROJECT_ROOT.parent / "data" / "pmdata" / "processed"

df = pd.read_parquet(PROCESSED_DATA_DIR / "pmdata_daily_features.parquet")
df.head()

,participant_id,date,steps_daily,sleep_minutes,time_in_bed,sleep_efficiency,hr_mean,hr_min,hr_max,hr_std,resting_heart_rate,calories_daily,mood,fatigue,stress,sleep_quality,readiness
0,p01,2019-11-01,17873.0,NaN,NaN,NaN,66.140075,46.0,140.0,15.080539,53.741074,4009.10,3.0,2.0,3.0,3.0,5.0
1,p01,2019-11-02,13118.0,378.0,430.0,97.0,62.602226,44.0,122.0,15.280065,52.881497,3533.56,3.0,2.0,3.0,3.0,6.0
2,p01,2019-11-03,14312.0,378.0,422.0,96.0,64.551841,45.0,132.0,14.532847,53.222024,3748.73,3.0,3.0,3.0,3.0,8.0
3,p01,2019-11-04,10970.0,361.0,399.0,96.0,63.552109,44.0,126.0,13.494872,54.311141,3353.38,3.0,3.0,3.0,3.0,8.0
4,p01,2019-11-05,16186.0,326.0,362.0,99.0,66.672530,44.0,163.0,23.635823,52.259110,3794.63,3.0,3.0,3.0,3.0,8.0


In [ ]:
target = "readiness"

feature_cols = [
    "steps_daily",
    "sleep_minutes",
    "time_in_bed",
    "sleep_efficiency",
    "hr_mean",
    "hr_min",
    "hr_max",
    "hr_std",
    "resting_heart_rate",
    "calories_daily",
    "mood",
]

model_df = df.dropna(subset=[target]).copy()
model_df = model_df.sort_values(["participant_id", "date"]).reset_index(drop=True)
model_df.head()


,participant_id,date,steps_daily,sleep_minutes,time_in_bed,sleep_efficiency,hr_mean,hr_min,hr_max,hr_std,resting_heart_rate,calories_daily,mood,fatigue,stress,sleep_quality,readiness
0,p01,2019-11-01,17873.0,NaN,NaN,NaN,66.140075,46.0,140.0,15.080539,53.741074,4009.10,3.0,2.0,3.0,3.0,5.0
1,p01,2019-11-02,13118.0,378.0,430.0,97.0,62.602226,44.0,122.0,15.280065,52.881497,3533.56,3.0,2.0,3.0,3.0,6.0
2,p01,2019-11-03,14312.0,378.0,422.0,96.0,64.551841,45.0,132.0,14.532847,53.222024,3748.73,3.0,3.0,3.0,3.0,8.0
3,p01,2019-11-04,10970.0,361.0,399.0,96.0,63.552109,44.0,126.0,13.494872,54.311141,3353.38,3.0,3.0,3.0,3.0,8.0
4,p01,2019-11-05,16186.0,326.0,362.0,99.0,66.672530,44.0,163.0,23.635823,52.259110,3794.63,3.0,3.0,3.0,3.0,8.0


In [10]:
eval_df = model_df.sample(30, random_state=42).copy()
eval_df = eval_df.reset_index(drop=True)
eval_df[["participant_id", "date", target]].head()

,participant_id,date,readiness
0,p06,2019-11-08,8.0
1,p11,2019-12-15,6.0
2,p06,2019-12-07,7.0
3,p16,2020-02-10,3.0
4,p11,2020-02-25,3.0


In [11]:
llm = get_ollama_llm(model="qwen2.5:1.5b", base_url="http://localhost:11434")

few_shot_examples = model_df.sample(3, random_state=42).to_dict(orient="records")

results_df = run_readiness_evaluation(
    eval_df=eval_df,
    llm=llm,
    few_shot_examples=few_shot_examples,
)

metrics = compute_readiness_metrics(results_df)

results_df.head(), metrics


(  participant_id        date  actual_readiness  predicted_readiness  \
 0            p06  2019-11-08               8.0                  8.0   
 1            p11  2019-12-15               6.0                  6.0   
 2            p06  2019-12-07               7.0                  7.0   
 3            p16  2020-02-10               3.0                  5.0   
 4            p11  2020-02-25               3.0                  7.0   
 
          raw_response  parse_success error_message prompt_version  
 0  {"readiness": 8.0}           True          None   readiness_v1  
 1  {"readiness": 6.0}           True          None   readiness_v1  
 2  {"readiness": 7.0}           True          None   readiness_v1  
 3  {"readiness": 5.0}           True          None   readiness_v1  
 4  {"readiness": 7.0}           True          None   readiness_v1  ,
 {'num_rows': 30,
  'num_valid_predictions': 30,
  'parse_failure_rate': np.float64(0.0),
  'mae': 2.066666666666667})